# Step 1：单个静态人群的部署

按 **问题 → 方法 → 实验 → 分析 → 局限** 阅读已经保存的结果。算法在 `src/udc/`，本笔记只调用包内入口和轨迹文件。

Step 1 固定为：一个静态人群、已知封闭场地、全局观测、无限通信、没有 crowd–guide 动力学交互。控制器只看见人员位置、半径和 demand 权重。demand 是规划权重，不是已经验证的风险。边界目前是人员圆盘的凸包。符号见 `docs/STEP1_THEORY.zh.md`。

## 问题

在固定 Guide 数量 $N$ 下，把站点放到长度为 $L$ 的闭合部署曲线上，使得：

1. 相邻弧间隙不超过规划上限 $H_p$；
2. 站点欧氏间距不小于安全阈值；
3. 目标对 Guide 初态可达；
4. 执行轨迹通过独立的九类验收。

空间热点提出的额外问题是：如何在热点对应的弧段提高部署密度，同时不破坏上述约束。$\lceil L/H_p\rceil$ 只是数量下界，不是完整可行性。搜索耗尽只说明有界搜索没找到方案。

## 方法

| 方法 | 曲线密度 | 约束处理 |
|---|---|---|
| Equal Arc | 均匀 | 直接等弧长构造 |
| Average CVT | 局部平均 demand | Lloyd；接受时检查间隙与间距 |
| Mass CVT | Gaussian 需求质量 | 同上 |
| DistMesh-inspired | $\sqrt{\rho}$ 分位松弛 | 没有间隙上限；失败由 $\sqrt{\rho}$ 不动点解释 |
| RA-ABDC 候选 | Mass 密度的幂次 $\rho^a$ | 每步投影到 $[d_{\min},H_p]$；不行则减小 $a$，最后记录 Equal Arc 回退 |

人员需求映射到曲线的唯一途径是 Gaussian 密度。空间热点是场景里的固定圆盘：背景需求恒为 1，圆内乘 3 后截断，初始化时冻结。导航、安全过滤和恢复层对五种方法相同。

## 实验

开发 seed 0–4。validation seeds 5000–5029 不运行。下面若已有结果就读取，否则只重跑 50 人案例。

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from udc.experiments.runner import experiment, reevaluate_result

ROOT = Path.cwd() if (Path.cwd() / 'configs' / 'step1' / 'static_50.yaml').exists() else Path.cwd().parent
STATIC = ROOT / 'runs' / 'pilot' / 'static50'
if not (STATIC / 'metrics.json').exists() and (ROOT / 'runs' / 'static50' / 'metrics.json').exists():
    STATIC = ROOT / 'runs' / 'static50'
if not (STATIC / 'metrics.json').exists():
    experiment(ROOT / 'configs' / 'step1' / 'static_50.yaml', STATIC, plots=False)
metrics = json.loads((STATIC / 'metrics.json').read_text(encoding='utf-8'))
checked = reevaluate_result(STATIC)
print(metrics['termination_status'], 'online', metrics['success'], 'reevaluated', checked['success'])
print('active guides', metrics['active_guides'], 'simulation time [s]', metrics['simulation_time'])
print('stage', metrics.get('stage_diagnosis'), 'demand', metrics.get('demand_kind'))

In [ ]:
trace = np.load(STATIC / 'trajectory.npz', allow_pickle=False)
summary = {
    'success': metrics['success'],
    'reevaluation_consistent': checked['criteria_consistent'],
    'simulation_time_s': metrics['simulation_time'],
    'actual_max_gap_m': metrics['actual_max_gap'],
    'mean_curve_distance_m': metrics['mean_curve_distance'],
    'path_length_m': metrics['path_length'],
    'minimum_net_clearances_m': metrics['minimum_net_clearances'],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
positions = trace['positions']
room = trace['room']
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, room[0], room[0], 0, 0], [0, 0, room[1], room[1], 0], color='#334155')
deployment = trace['boundary_deployment']
ax.plot(*np.vstack([deployment, deployment[0]]).T, color='#0f766e', label='deployment curve')
ax.scatter(trace['crowd_positions'][:, 0], trace['crowd_positions'][:, 1], s=12, c='#b45309', label='frozen crowd')
for guide in range(positions.shape[1]):
    ax.plot(positions[:, guide, 0], positions[:, guide, 1], color='#1d4ed8', lw=1)
ax.scatter(positions[-1, :, 0], positions[-1, :, 1], c='#1d4ed8', label='guide arrival')
ax.set_aspect('equal'); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_title('Static 50-person Equal Arc'); ax.legend(loc='upper right', fontsize=8)
fig.tight_layout(); plt.show()

配对试点命令（不要在笔记里重跑 150 个任务）：

```bash
python -m udc.experiments.comparison --hotspot --output runs/pilot/high_demand_20_24 --workers 8
python -m udc.experiments.comparison --spec experiments/step1_hotspot_pilot.yaml --output runs/pilot/hotspot --workers 8
```

三档数量 14／17／19 是相对生成多边形下界 15 的偏移，实现凸包下界是 14。它们不是充足／临界／不足的标签。

## 分析

成功率分母包含全部失败。规划目标指标与轨迹指标分开。五个开发 seed 只用于诊断。

In [ ]:
pilot = ROOT / 'runs' / 'pilot' / 'hotspot' / 'comparison_summary.json'
diag = ROOT / 'runs' / 'pilot' / 'high_demand_20_24' / 'diagnosis.json'
if diag.exists():
    report = json.loads(diag.read_text(encoding='utf-8'))
    print('20/24 pairing', report['pairing'])
    for row in report['runs']:
        print(row['fixed_n'], row['method'], row['stage_diagnosis'], row['termination_status'],
              'plan', row['plan_found'], 'assign', row['assignment_found'])
if pilot.exists():
    summary = json.loads(pilot.read_text(encoding='utf-8'))
    print('pilot', summary['tasks'], 'successful', summary['successful'], 'pairing', summary['pairing']['consistent'])
    for key, group in sorted(summary['groups'].items()):
        print(f"{group['cell']:42} {group['method']:12} {group['successful']}/{group['tasks']} {group['stages']}")
else:
    print('热点试点结果不在 runs/pilot/hotspot；请先按上面的命令运行。')

In [ ]:
figures = ROOT / 'runs' / 'pilot' / 'hotspot' / 'figures'
for name in [
    'ra_abdc_hotspot_moderate_seed0.png',
    'equal_arc_hotspot_moderate_seed0.png',
    'ra_abdc_hotspot_tight_seed0.png',
    'comparison_summary.png',
]:
    path = figures / name if name != 'comparison_summary.png' else ROOT / 'runs' / 'pilot' / 'hotspot' / name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))

当前 150 任务诊断里：Equal Arc 与 RA-ABDC 各 30/30 成功；Average CVT 25/30；Mass CVT 与 DistMesh-inspired 各 20/30。全部失败都是 `no_plan_found`，必要数量条件成立，因此不能写成“不存在可行方案”。N=17／19 时 RA-ABDC 与 Mass CVT 的间隙、代价和热点浓度相同。N=14 热点时投影把间隙顶在 1.94 m，热点浓度略高于 Equal Arc。DistMesh 的宽间隙来自 $\sqrt{\rho}$ 不动点，本轮没有调参。

## 局限

- 50 人网格是确定性工程复现，不是独立随机样本。
- 五个开发 seed 的 150 任务只用于诊断，不能作为方法排序或显著性结论。
- 完整因子表（现为 4050 任务）没有运行。
- 凸包会填平凹陷；demand 不是经验证风险；RA-ABDC 不是风险控制算法。
- 约束未激活时，名称不同的方法可以给出同一组目标。
- Step 2／3（动态多群、有限感知与通信）没有在本轮实现。

是否扩大场景或进入正式实验，取决于这份受控试点是否已经回答当前的规划可行性问题。